In [8]:
import pandas as pd

In [9]:
df = pd.read_csv(r'C:\Users\Shubham\Downloads\RetailCleaned.csv' , header=None)

C:\Users\Shubham\AppData\Local\Temp\ipykernel_13564\3734402028.py:1: DtypeWarning: Columns (0: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\Shubham\Downloads\RetailCleaned.csv' , header=None)


In [8]:
df.head()

,0,1,2,3,4,5,6,7,8
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00.0000000,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00.0000000,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00.0000000,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00.0000000,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00.0000000,1.65,13085.0,United Kingdom,2010-2011


In [10]:
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country','SourceSheet']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()

(1033036, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  str           
 8   SourceSheet  1033036 non-null  str           
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), str(4)
memory usage: 70.9+ MB


In [11]:
#filter and calculate Recency , Frequency & Monetary 
rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary = ('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()


In [13]:
#Score Customers and Assign Business Segments
rfm['R_Score'] = pd.qcut(rfm['Recency'],5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],5, labels=[1,2,3,4,5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Need Attention'
    else:
        return 'Lost'
    
rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90



In [14]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Shubham\Downloads\RetailCleaned.csv' , header=None)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

df = df[(df["Customer_ID"].notna()) & (df["Quantity"] > 0)].copy()
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
df["Revenue"] = df["Quantity"] * df["Price"]

df_final = df.merge(
    rfm[[
        "Customer_ID", "Recency", "Frequency", "Monetary",
        "R_Score", "F_Score", "M_Score", "Segment", "IsChurned"
    ]],
    on="Customer_ID",
    how="left"
)

df_final.to_csv("EcommerceDashboardData.xlsx", index=False)